# TF-IDF + Logistic Regression Baseline

Run this notebook from the project root or from the `baselines/` folder. It uses the shared benchmark code, writes artifacts under `artefacts/`, and evaluates `tfidf_logreg` on the fixed split.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "baselines").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

PosixPath('/home/ilya/ML/NLP/project')

In [2]:
MODEL_NAME = "tfidf_logreg"
MAX_SAMPLES = None  # set to a small integer, e.g. 3000, for debugging
TOP_GENRES = 15
EPOCHS = 3
BATCH_SIZE = 16
TFIDF_MAX_FEATURES = 100_000

In [3]:
import pandas as pd

from baselines.config import BaselineConfig
from baselines.data_utils import prepare_data
from baselines.run_all import finalize_model_result
from baselines.sklearn_baselines import run_tfidf_logreg

config = BaselineConfig(
    max_samples=MAX_SAMPLES,
    top_genres=TOP_GENRES,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    tfidf_max_features=TFIDF_MAX_FEATURES,
)

In [4]:
bundle = prepare_data(config)
bundle.stats

Found candidate tabular files:
  /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv (80.5 MB)
  /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.parquet (50.9 MB)
Choosing largest file by default: /home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv
Prepared data: train=87575, val=34470, test=32986, labels=15, split=temporal_train_le_2023_val_2024_test_2025


{'source_path': '/home/ilya/ML/NLP/project/data/tmdb_movies_2021_2025.csv',
 'detected_columns': {'title': 'title',
  'overview': 'overview',
  'genres': 'genres',
  'release_date': 'release_date',
  'id': 'tmdb_id'},
 'rows_before_filtering': 232586,
 'rows_after_overview_filter': 193927,
 'rows_after_genre_parse': 155678,
 'rows_after_top_genre_filter': 155031,
 'selected_genre_labels': ['Drama',
  'Documentary',
  'Comedy',
  'Horror',
  'Thriller',
  'Animation',
  'Romance',
  'Music',
  'Action',
  'Crime',
  'Fantasy',
  'Science Fiction',
  'Mystery',
  'Family',
  'TV Movie'],
 'per_label_frequency': {'Drama': 54784,
  'Documentary': 43107,
  'Comedy': 29457,
  'Horror': 18098,
  'Thriller': 15386,
  'Animation': 12723,
  'Romance': 10576,
  'Music': 8382,
  'Action': 7238,
  'Crime': 6677,
  'Fantasy': 6339,
  'Science Fiction': 6141,
  'Mystery': 6245,
  'Family': 5088,
  'TV Movie': 3842},
 'train_per_label_frequency': {'Drama': 30116,
  'Documentary': 25421,
  'Comedy': 15

In [5]:
assert bundle.y_train.shape[1] == len(bundle.label_names)
assert bundle.y_val.shape[1] == len(bundle.label_names)
assert bundle.y_test.shape[1] == len(bundle.label_names)
assert {"sample_id", "text", "labels_list"}.issubset(bundle.train_df.columns)
assert bundle.train_df["text"].str.len().min() >= config.min_overview_chars

print("labels:", bundle.label_names)
print("train/val/test:", bundle.y_train.shape, bundle.y_val.shape, bundle.y_test.shape)

labels: ['Drama', 'Documentary', 'Comedy', 'Horror', 'Thriller', 'Animation', 'Romance', 'Music', 'Action', 'Crime', 'Fantasy', 'Science Fiction', 'Mystery', 'Family', 'TV Movie']
train/val/test: (87575, 15) (34470, 15) (32986, 15)


In [6]:
result = run_tfidf_logreg(bundle, config)
metrics = finalize_model_result(result, bundle, config)
metrics

{'model': 'tfidf_logreg',
 'micro_f1': 0.5774017268704155,
 'macro_f1': 0.4708204899504082,
 'weighted_f1': 0.5796793614747219,
 'samples_f1': 0.5955500536368177,
 'precision_micro': 0.5206553485248411,
 'recall_micro': 0.6480307919598771,
 'hamming_loss': 0.09862163746235776,
 'precision_at_1': 0.658400533559692,
 'precision_at_3': 0.37766527213565343,
 'recall_at_3': 0.7803062434375719,
 'subset_accuracy': 0.2792093615473231,
 'average_precision_micro': 0.572045556356131}

In [7]:
prediction_path = PROJECT_ROOT / "artefacts" / "predictions" / f"{MODEL_NAME}_test_predictions.csv"
threshold_path = PROJECT_ROOT / "artefacts" / "thresholds" / f"{MODEL_NAME}_thresholds.json"
print(prediction_path)
print(threshold_path)
pd.read_csv(prediction_path).head()

/home/ilya/ML/NLP/project/artefacts/predictions/tfidf_logreg_test_predictions.csv
/home/ilya/ML/NLP/project/artefacts/thresholds/tfidf_logreg_thresholds.json


,sample_id,text,true_labels,predicted_labels,score_drama,score_documentary,score_comedy,score_horror,score_thriller,score_animation,score_romance,score_music,score_action,score_crime,score_fantasy,score_science_fiction,score_mystery,score_family,score_tv_movie
0,1052558,iPossessed [SEP] A group of celebrating friend...,Horror|Thriller,Horror|Thriller|Fantasy,0.171239,0.044405,0.310517,0.986476,0.848536,0.128298,0.128184,0.133134,0.401488,0.263059,0.777477,0.192024,0.291670,0.072677,0.091326
1,980477,Ne Zha 2 [SEP] After a catastrophic event leav...,Animation|Action|Fantasy,Drama|Action|Fantasy|Science Fiction,0.721349,0.045785,0.409116,0.079950,0.207994,0.198348,0.558181,0.208357,0.767181,0.127458,0.590311,0.720377,0.073796,0.135551,0.042957
2,1205229,Night of the Zoopocalypse [SEP] A wolf and mou...,Comedy|Horror|Animation|Science Fiction,Horror|Animation|Action|Science Fiction,0.046691,0.114213,0.388555,0.830691,0.440229,0.847745,0.053184,0.054573,0.930710,0.125339,0.325064,0.683831,0.048124,0.421554,0.403499
3,1084199,Companion [SEP] During a weekend getaway at a ...,Horror|Thriller|Science Fiction,Drama|Thriller|Mystery,0.580751,0.091227,0.460285,0.561122,0.856499,0.028010,0.304731,0.097233,0.228660,0.405865,0.053360,0.548916,0.657229,0.019067,0.094253
4,1009640,Valiant One [SEP] With tensions between North ...,Thriller|Action,Action,0.286793,0.339684,0.304885,0.169217,0.302474,0.233757,0.265517,0.043970,0.703803,0.202967,0.117772,0.329611,0.107750,0.094373,0.178513
